# Loan Dataset Cleaning — Step by Step

This notebook documents the cleaning process applied to the loan dataset.

### Cleaning tasks
1. Load and inspect the dataset.
2. Check data types.
3. Check missing values.
4. Check duplicate records.
5. Inspect categorical values for inconsistent formats/typos.
6. Standardise inconsistent categorical values.
7. Convert numeric columns to the correct data types.
8. Handle missing values.
9. Remove duplicate records.
10. Run final validation checks.
11. Export the cleaned dataset.


In [ ]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv("1f79ac1d-9aca-4537-ae2f-e424148eb8c5.csv")

# Create a copy so the original dataframe remains unchanged
data = df.copy()

data.head()


## 1. Inspect the dataset

First, check the number of rows and columns and look at the column names.

In [ ]:
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
print("\nColumn names:")
print(data.columns.tolist())


## 2. Check the data types

This helps identify columns that may have been stored using the wrong data type.

In [ ]:
data.dtypes


## 3. Check for missing values

Missing values are counted for every column so we can decide how to handle them.

In [ ]:
missing_values = data.isna().sum().sort_values(ascending=False)
missing_values


## 4. Check for duplicate records

Duplicate rows can cause the same loan application to be counted more than once.

In [ ]:
duplicate_count = data.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)


## 5. Inspect categorical columns

The categorical columns contain inconsistent representations. For example, `Gender` contains both `Male`/`Female` and `m`/`f`, while `Property_Area` contains several spelling variations.

In [ ]:
categorical_cols = [
    "Gender", "Married", "Dependents",
    "Education", "Self_Employed", "Property_Area"
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(data[col].value_counts(dropna=False))


## 6. Standardise the Gender column

The values `m` and `f` are alternative formats for `Male` and `Female`. They are converted to one consistent format.

In [ ]:
data["Gender"] = data["Gender"].replace({
    "m": "Male",
    "f": "Female"
})

data["Gender"].value_counts(dropna=False)


## 7. Correct spelling variations in Property_Area

Several values represent the same three property-area categories but contain spelling errors.

They are mapped to:
- `Urban`
- `Semiurban`
- `Rural`


In [ ]:
property_area_mapping = {
    "Semiurbn": "Semiurban",
    "Semurban": "Semiurban",
    "Semiurben": "Semiurban",
    "Uban": "Urban",
    "Urben": "Urban",
    "Urbun": "Urban",
    "Urbn": "Urban",
    "Rurl": "Rural",
    "Rurall": "Rural",
    "Rurel": "Rural",
    "Rual": "Rural"
}

data["Property_Area"] = data["Property_Area"].replace(property_area_mapping)

data["Property_Area"].value_counts(dropna=False)


## 8. Convert numeric columns to numeric data types

The following columns should contain numbers:
- `ApplicantIncome`
- `CoapplicantIncome`
- `LoanAmount`
- `Loan_Amount_Term`
- `Credit_History`

`errors="coerce"` converts any invalid numeric entries to missing values instead of causing the cleaning process to fail.

In [ ]:
numeric_cols = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data[numeric_cols].dtypes


## 9. Recheck missing values

After the standardisation and type conversion steps, check the remaining missing values.

In [ ]:
data.isna().sum().sort_values(ascending=False)


## 10. Fill missing categorical values

For categorical variables, the mode (the most common value) is used.

This is appropriate here because these fields contain categories rather than continuous measurements.

In [ ]:
for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

data[categorical_cols].isna().sum()


## 11. Fill missing numeric values

- `LoanAmount` is filled using its median.
- `Loan_Amount_Term` is filled using its median.
- `Credit_History` is filled using its mode because it is a binary field (`0`/`1`).

Median is used for the continuous/ordered numeric fields because it is less affected by extreme values than the mean.

In [ ]:
data["LoanAmount"] = data["LoanAmount"].fillna(data["LoanAmount"].median())
data["Loan_Amount_Term"] = data["Loan_Amount_Term"].fillna(data["Loan_Amount_Term"].median())
data["Credit_History"] = data["Credit_History"].fillna(data["Credit_History"].mode()[0])

data[["LoanAmount", "Loan_Amount_Term", "Credit_History"]].isna().sum()


## 12. Remove duplicate records

Remove any duplicate rows and reset the index.

If the duplicate check above returned `0`, this step will not remove any rows.

In [ ]:
before_duplicates = len(data)

data = data.drop_duplicates().reset_index(drop=True)

after_duplicates = len(data)

print("Rows before:", before_duplicates)
print("Rows after:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)


## 13. Finalise numeric formats

The loan term and credit history are whole-number fields, while income may contain decimal values. `CoapplicantIncome` is rounded to two decimal places.

`LoanAmount` is converted to an integer because all observed non-missing values are whole numbers.

In [ ]:
data["ApplicantIncome"] = data["ApplicantIncome"].astype(int)
data["CoapplicantIncome"] = data["CoapplicantIncome"].round(2)
data["LoanAmount"] = data["LoanAmount"].round().astype(int)
data["Loan_Amount_Term"] = data["Loan_Amount_Term"].round().astype(int)
data["Credit_History"] = data["Credit_History"].astype(int)

data.dtypes


## 14. Final validation

Check that:
- there are no missing values,
- there are no duplicate rows,
- categorical values are standardised,
- numeric columns have the expected data types.

In [ ]:
print("Missing values:")
print(data.isna().sum())

print("\nDuplicate rows:", data.duplicated().sum())

print("\nGender values:")
print(data["Gender"].unique())

print("\nProperty_Area values:")
print(data["Property_Area"].unique())

print("\nFinal data types:")
print(data.dtypes)


## 15. Compare the dataset before and after cleaning

In [ ]:
print("Original shape:", df.shape)
print("Cleaned shape:", data.shape)

print("\nOriginal missing values:", df.isna().sum().sum())
print("Cleaned missing values:", data.isna().sum().sum())

print("\nOriginal duplicates:", df.duplicated().sum())
print("Cleaned duplicates:", data.duplicated().sum())


## 16. View the cleaned dataset

In [ ]:
data.head(10)


## 17. Export the cleaned dataset

The cleaned data is saved as a new CSV file so the original raw dataset is preserved.

In [ ]:
data.to_csv("loan_dataset_cleaned.csv", index=False)

print("Cleaned dataset saved as: loan_dataset_cleaned.csv")


# Cleaning Summary

The dataset was cleaned by:

- standardising inconsistent `Gender` values (`m`/`f` → `Male`/`Female`);
- correcting spelling variations in `Property_Area`;
- converting numeric fields to numeric data types;
- filling missing categorical values with the mode;
- filling `LoanAmount` and `Loan_Amount_Term` missing values with their medians;
- filling missing `Credit_History` values with its mode;
- checking and removing duplicate records;
- finalising numeric formats;
- validating the cleaned dataset;
- exporting the final dataset as `loan_dataset_cleaned.csv`.

No duplicate rows were present in the original dataset.
